<a href="https://colab.research.google.com/github/lunecarvalho/newslens-project/blob/main/04_ner_lemmatizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv('/content/dataset_preprocessado.csv')

print("Dimensões:", df.shape)
print("Colunas:")
print(df.columns.tolist())

Dimensões: (7199, 10)
Colunas:
['id', 'label', 'arquivo', 'texto_original', 'texto_preprocessado', 'topico_bertopic', 'tema', 'tamanho_caracteres', 'tamanho_palavras', 'texto_preprocessado_sem_acento']


In [37]:
df.head()

,id,label,arquivo,texto_original,texto_preprocessado,topico_bertopic,tema,tamanho_caracteres,tamanho_palavras,texto_preprocessado_sem_acento,texto_lematizado
0,1,falso,1170.txt,Falso médico é preso pela PM de Santa Catarina...,falso médico é preso pela pm de santa catarina...,28,Saúde,881,149,falso medico e preso pela pm de santa catarina...,falso medico e prender por o pm de santa catar...
1,2,falso,1688.txt,Antes da confirmação da morte de Teori Zavasck...,antes da confirmação da morte de teori zavasck...,0,Política,711,121,antes da confirmacao da morte de teori zavasck...,antes de o confirmacao de o morte de teori zav...
2,3,falso,1863.txt,Senador que assume o lugar de Renan foi flagra...,senador que assume o lugar de renan foi flagra...,55,Política,3108,555,senador que assume o lugar de renan foi flagra...,senador que assumir o lugar de renan ser flagr...
3,4,falso,1563.txt,"Repórter relata conversa entre deputados: ""A D...",repórter relata conversa entre deputados: a di...,3,Corrupção,1674,284,reporter relata conversa entre deputados: a di...,reporter relato conversa entre deputado : o di...
4,5,falso,3556.txt,Governo repressor e comunista usa exército par...,governo repressor e comunista usa exército par...,12,Política,724,113,governo repressor e comunista usa exercito par...,governo repressor e comunista usar exercito pa...


In [3]:
!pip install -q spacy
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 60.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import spacy

print(spacy.__version__)

nlp = spacy.load("pt_core_news_sm")

print(nlp.pipe_names)

3.8.16
['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']


In [5]:
import spacy

nlp = spacy.load(
    "pt_core_news_sm",
    disable=["parser", "ner"]
)

In [6]:
texto_teste = "A lematização em Processamento de Linguagem Natural (NLP) é uma técnica de pré-processamento que converte palavras flexionadas em sua forma base e dicionarizada, chamada de lema."

doc = nlp(texto_teste)

[(token.text, token.lemma_) for token in doc]

[('A', 'o'),
 ('lematização', 'lematização'),
 ('em', 'em'),
 ('Processamento', 'Processamento'),
 ('de', 'de'),
 ('Linguagem', 'Linguagem'),
 ('Natural', 'Natural'),
 ('(', '('),
 ('NLP', 'NLP'),
 (')', ')'),
 ('é', 'ser'),
 ('uma', 'um'),
 ('técnica', 'técnica'),
 ('de', 'de'),
 ('pré-processamento', 'pré-processamento'),
 ('que', 'que'),
 ('converte', 'converter'),
 ('palavras', 'palavra'),
 ('flexionadas', 'flexionar'),
 ('em', 'em'),
 ('sua', 'seu'),
 ('forma', 'forma'),
 ('base', 'base'),
 ('e', 'e'),
 ('dicionarizada', 'dicionarizar'),
 (',', ','),
 ('chamada', 'chamar'),
 ('de', 'de'),
 ('lema', 'lema'),
 ('.', '.')]

In [20]:
def lematizar_texto(texto):
    doc = nlp(texto)

    lemas = []

    for token in doc:
        if token.is_space:
            continue

        # Corrige nomes próprios que o modelo pode lematizar incorretamente
        if token.text.lower() == "dilma":
            lema = "dilma"

        elif token.text.lower() == "rousseff":
            lema = "rousseff"

        # Preserva entidades reconhecidas pelo NER
        elif token.ent_type_:
            lema = token.text.lower()

        else:
            lema = token.lemma_.lower()

        lemas.append(lema)

    return " ".join(lemas)

In [21]:
documentos = df["texto_original"].fillna("").tolist()

textos_lematizados = []

for doc in nlp.pipe(documentos, batch_size=50):

    lemas = []

    for token in doc:

        if token.is_space:
            continue

        # Preserva entidades nomeadas
        if token.ent_type_:
            lema = token.text.lower()

        else:
            lema = token.lemma_.lower()

        lemas.append(lema)

    textos_lematizados.append(" ".join(lemas))

df["texto_lematizado"] = textos_lematizados

In [22]:
df["texto_lematizado"] = df["texto_preprocessado_sem_acento"].apply(
    lematizar_texto
)

In [23]:
print(
    "dilmar:",
    df["texto_lematizado"].str.contains(
        "dilmar",
        case=False,
        na=False
    ).sum()
)

print(
    "dilma:",
    df["texto_lematizado"].str.contains(
        "dilma",
        case=False,
        na=False
    ).sum()
)

dilmar: 1
dilma: 1094


In [24]:
resultado = df[
    df["texto_lematizado"].str.contains(
        "dilmar",
        case=False,
        na=False
    )
]

print(resultado[[
    "id",
    "texto_preprocessado_sem_acento",
    "texto_lematizado"
]].to_string(index=False))

  id                                                                                                                                                                                                                                                                                                                                                                                                                                             texto_preprocessado_sem_acento                                                                                                                                                                                                                                                                                                                                                                                                                                                               texto_lematizado
2109 dilmarvore: dilma se compara a uma arvore e diz que esta sofrendo um

In [25]:
print(
    df.loc[
        df["texto_lematizado"].str.contains("dilmar", case=False, na=False),
        "texto_lematizado"
    ].iloc[0]
)

dilmarvore : dilma se comparar a um arvore e dizer que este sofrer um ataque de fungo . dilma dizer que her um instabilidade juridico em o pai . so para lembra-la , todo o processo de impeachment este amparar por o lei e ter o crivo de o supremo tribunal federal . dilma insistir em dizer que nao ha base juridico para que ela ser afastar e explicar : e como se voce derrubar um arvore . voce derrubar o arvore como se ela sofrer um ataque de fungo , de parasita


In [11]:
nomes = ["lula", "temer", "bolsonaro", "moro", "dilma"]

for nome in nomes:
    quantidade = df["texto_lematizado"].str.contains(
        rf"\b{nome}\b",
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f"{nome}: {quantidade}")

lula: 1593
temer: 1190
bolsonaro: 166
moro: 655
dilma: 1094


In [16]:
df["texto_lematizado"].head(10)

,texto_lematizado
0,falso medico e prender por o pm de santa catar...
1,antes de o confirmacao de o morte de teori zav...
2,senador que assumir o lugar de renan ser flagr...
3,reporter relato conversa entre deputado : o di...
4,governo repressor e comunista usar exercito pa...
5,urgente : justica federal de brasilia receber ...
6,"tico santa cruz , o papagaio de pirata de o pt..."
7,jornalista dizer que 3 senador de o pt querer ...
8,dilmar querer transformar defesa em o senado e...
9,alemanha voltar a fazer alerta e pedir para po...


In [26]:
print(df["texto_lematizado"].head(10).to_string())

0    falso medico e prender por o pm de santa catar...
1    antes de o confirmacao de o morte de teori zav...
2    senador que assumir o lugar de renan ser flagr...
3    reporter relato conversa entre deputado : o di...
4    governo repressor e comunista usar exercito pa...
5    urgente : justica federal de brasilia receber ...
6    tico santa cruz , o papagaio de pirata de o pt...
7    jornalista dizer que 3 senador de o pt querer ...
8    dilma querer transformar defesa em o senado em...
9    alemanha voltar a fazer alerta e pedir para po...


In [31]:
print(df.shape)

print(df[[
    "texto_preprocessado_sem_acento",
    "texto_lematizado"
]].head(5).to_string())

(7199, 11)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [33]:
print(df.columns.tolist())

['id', 'label', 'arquivo', 'texto_original', 'texto_preprocessado', 'topico_bertopic', 'tema', 'tamanho_caracteres', 'tamanho_palavras', 'texto_preprocessado_sem_acento', 'texto_lematizado']


In [34]:
df.to_csv(
    "dataset_preprocessado.csv",
    index=False,
    encoding="utf-8-sig"
)

In [35]:
import os

print(
    "Arquivo criado:",
    os.path.exists("dataset_preprocessado.csv")
)

print(
    "Tamanho:",
    os.path.getsize("dataset_preprocessado.csv"),
    "bytes"
)

Arquivo criado: True
Tamanho: 32896138 bytes


In [36]:
from google.colab import files

files.download("dataset_preprocessado.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>